# Fine-tuning Qwen2.5 for cover-letter generation

Use a CUDA Google Colab runtime. Upload `cover_letter_dataset_3000.json` to the working directory first.

In [ ]:
!pip install -q unsloth trl peft accelerate bitsandbytes datasets


In [ ]:
import json
from pathlib import Path

import torch
from datasets import Dataset

DATASET_PATH = Path("cover_letter_dataset_3000.json")
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"{DATASET_PATH} was not found. Upload it to Colab first.")

with DATASET_PATH.open(encoding="utf-8") as dataset_file:
    records = json.load(dataset_file)

if not isinstance(records, list) or not all("instruction" in item and "input" in item and "output" in item for item in records):
    raise ValueError("Dataset must be a JSON list with instruction, input, and output fields.")

print(f"Loaded {len(records)} examples")
print(records[1] if len(records) > 1 else records[0])
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime before fine-tuning.")


In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model, r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=128, lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=3407,
    use_rslora=False, loftq_config=None,
)


In [ ]:
# Training and inference use the same Qwen chat template.
def format_example(example):
    user_content = f"{example['instruction']}\n\n{example['input']}"
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

dataset = Dataset.from_dict({"text": [format_example(item) for item in records]})
print(dataset[0]["text"])


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=10, num_train_epochs=4, learning_rate=1e-4,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=3407, output_dir="outputs",
        save_strategy="epoch", save_total_limit=2, dataloader_pin_memory=False, report_to="none",
    ),
)
trainer_stats = trainer.train()
print(trainer_stats)


In [ ]:
FastLanguageModel.for_inference(model)

instruction = (
    "На основе текста вакансии и информации о проектах кандидата напиши "
    "персонализированное сопроводительное письмо на русском языке. "
    "Используй только факты из вакансии и проектов. Не придумывай опыт, "
    "навыки, цифры, компании или результаты. Если совпадения нет, не упоминай его."
)
candidate_data = """job_title: Fullstack-developer
job_text: Ищем Fullstack-разработчика с опытом LLM, Vue 3, PHP, REST API, RAG и интеграций amoCRM, Calltouch, Telegram.

projects:
- Система мониторинга технических компетенций: PHP, MySQL, Redis, Vue, JavaScript, Laravel.
- Lidofon.ru — CRM для call-центра: PHP, Laravel, PostgreSQL, Redis, Docker.
- SmartEat — приложение здорового питания: Python, Django, Typesense, Firebase, OpenAI, CI/CD.
"""

# The training samples use one user message: instruction followed by the source data.
messages = [{"role": "user", "content": f"{instruction}\n\n{candidate_data}"}]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=256, use_cache=True, do_sample=False, repetition_penalty=1.05)
print(tokenizer.batch_decode(outputs, skip_special_tokens=False)[0])


In [ ]:
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

from google.colab import files

gguf_files = sorted(Path("gguf_model").glob("*.gguf"))
if not gguf_files:
    raise FileNotFoundError("No GGUF file was produced in gguf_model.")
print(f"Downloading: {gguf_files[0]}")
files.download(str(gguf_files[0]))
